In [ ]:
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
import torch
import gensim.downloader
import pandas as pd
import spacy
from collections import defaultdict
import random

In [ ]:
def load_data(path):
    dataset = load_dataset("wikitext", "wikitext-103-v1", split="train")
    glove = gensim.downloader.load("glove-wiki-gigaword-50")
    df = pd.read_csv(path, 
                sep="\t", 
                names=["lemma", "forms", "fam", "transf"])
    return dataset, glove, df

In [ ]:
def clean_dataset(dataset):
    content = []
    for line in dataset["text"]:
        line = line.strip().lower()
        if line and not line.startswith("=") and len(line.split()) > 4:
            content.append(line)

In [ ]:
def get_target_words(df, glove):
    glove_voc = set(glove.key_to_index.keys())
    target = {
            word.strip().lower()
            for form_list in df["forms"].str.split(",")
            for word in form_list
            if word.strip().lower() in glove_voc
    }
    return target

In [ ]:
def get_matched_sent(content, target):
    nlp = spacy.blank("en")
    nlp.add_pipe("sentencizer")

    matched = defaultdict(set)

    for doc in nlp.pipe(content, batch_size=2048, n_process=2):
        for sent in doc.sents:
            if len(sent) > 40:
                continue

            sent_text = sent.text
            tokens = {token.text for token in sent}

            for word in target.intersection(tokens):
                matched[word].append(sent_text)
    return matched

In [ ]:
def get_sent_df(matched, json_path=None):
    if json_path:
        return pd.read_json(json_path)

    def tag_word(sents):
        n = len(sents)
        if n >= 50:
            return "sampled"
        elif n >= 10:
            return "low_freq"
        return "excluded"

    def sample_sents(sents):
        sents = list(sents)
        n = len(sents)
        if n >= 50:
            return random.sample(sents, 50)
        return sents

    random.seed(42)
    sent_df = pd.DataFrame(matched.items(),
                            columns=["word", "sents"])

    sent_df["tag"] = sent_df["sents"].apply(tag_word)
    sent_df["sents"] = sent_df.apply(sample_sents, axis="columns")

    sent_df.to_json("sent_df.json")
    print("sent_df json created")
    
    return sent_df

In [ ]:
dataset, glove_voc, df = load_data("/home/onyxia/work/morph_families.tsv")

content = clean_dataset(dataset)
del dataset

target = get_target_words(df, glove_voc)
del df

matched = get_matched_sent(content, target)
del content

sent_df = get_sent_df(matched)